In [1]:
import os
os.getcwd()


'c:\\Users\\Aditya\\OneDrive\\Desktop\\UIDAI_DATA'

## cleaning

### libraries imported

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import plotly.express as px
import requests


### enrolment

In [3]:
enrol_files = glob.glob("enrolment/*.csv")

enrol_list = [pd.read_csv(f) for f in enrol_files]
enrol = pd.concat(enrol_list, ignore_index=True)


In [4]:
enrol.shape
enrol.columns


Index(['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17',
       'age_18_greater'],
      dtype='object')

### demographic

In [5]:
demo_files = glob.glob("demographic/*.csv")

demo_list = [pd.read_csv(f) for f in demo_files]
demo = pd.concat(demo_list, ignore_index=True)


In [6]:
demo.shape
demo.columns


Index(['date      ', ' state                                   ',
       ' district                       ', ' pincode', ' demo_age_5_17',
       ' demo_age_17_', 'date', 'state', 'district', 'pincode',
       'demo_age_5_17', 'demo_age_17_'],
      dtype='object')

In [7]:
for df in [enrol, demo]:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )


In [8]:
enrol.head()
demo.head()


,date,state,district,pincode,demo_age_5_17,demo_age_17_,date,state,district,pincode,demo_age_5_17,demo_age_17_
0,01-03-2025,Uttar Pradesh,Gorakhpur,273213.0,49.0,529.0,NaN,NaN,NaN,NaN,NaN,NaN
1,01-03-2025,Andhra Pradesh,Chittoor,517132.0,22.0,375.0,NaN,NaN,NaN,NaN,NaN,NaN
2,01-03-2025,Gujarat,Rajkot,360006.0,65.0,765.0,NaN,NaN,NaN,NaN,NaN,NaN
3,01-03-2025,Andhra Pradesh,Srikakulam,532484.0,24.0,314.0,NaN,NaN,NaN,NaN,NaN,NaN
4,01-03-2025,Rajasthan,Udaipur,313801.0,45.0,785.0,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
enrol['total_enrolments'] = (
    enrol['age_0_5'] +
    enrol['age_5_17'] +
    enrol['age_18_greater']
)


In [10]:
demo.columns = demo.columns.str.strip().str.replace(' ', '_')
demo = demo.loc[:, ~demo.columns.duplicated()].copy()

demo['total_updates'] = (
    pd.to_numeric(demo['demo_age_5_17'], errors='coerce').fillna(0) +
    pd.to_numeric(demo['demo_age_17_'], errors='coerce').fillna(0)
)


In [11]:
enrol_state = enrol.groupby('state')['total_enrolments'].sum().reset_index()


In [12]:
demo_state = demo.groupby('state')['total_updates'].sum().reset_index()


In [13]:
state_data = pd.merge(
    enrol_state,
    demo_state,
    on='state',
    how='inner'
)


In [14]:
state_data.head()


,state,total_enrolments,total_updates


In [15]:
state_data['administrative_burden_index'] = (
    state_data['total_updates'] /
    state_data['total_enrolments']
)


In [16]:
state_data.replace([np.inf, -np.inf], np.nan, inplace=True)
state_data.dropna(inplace=True)


In [17]:
state_data.sort_values(
    by='administrative_burden_index',
    ascending=False
).head(10)


,state,total_enrolments,total_updates,administrative_burden_index


## visualization

### Aadhaar Maintenance Stress Across Indian States

In [18]:
# 1. Remove duplicate columns (critical safety step)
enrol_df = enrol.loc[:, ~enrol.columns.duplicated()].copy()
demo_df  = demo.loc[:,  ~demo.columns.duplicated()].copy()

# 2. Clean state columns
enrol_df['state'] = enrol_df['state'].astype(str).str.strip()
demo_df['state']  = demo_df['state'].astype(str).str.strip()

enrol_df = enrol_df[enrol_df['state'].str.lower() != 'nan']
demo_df  = demo_df[demo_df['state'].str.lower() != 'nan']

# 3. Aggregate totals by state
enrol_state = enrol_df.groupby('state', as_index=False)['total_enrolments'].sum()
demo_state  = demo_df.groupby('state',  as_index=False)['demo_age_17_'].sum()

# 4. Merge datasets
state_data = pd.merge(enrol_state, demo_state, on='state', how='inner')

# 5. Compute maintenance stress
state_data['updates_per_100'] = (
    state_data['demo_age_17_'] / state_data['total_enrolments']
) * 100

# 6. Take top stressed states
state_data = state_data.sort_values(
    'updates_per_100', ascending=False
).head(15)

# 7. Plot (CLEAN & JUDGE-FRIENDLY)
fig1 = px.bar(
    state_data,
    x='updates_per_100',
    y='state',
    orientation='h',
    title="Aadhaar Maintenance Stress by State",
    labels={
        'updates_per_100': 'Updates per 100 Enrolments',
        'state': 'State'
    },
    color='updates_per_100',
    color_continuous_scale='YlOrRd'
)

fig1.update_layout(
    yaxis=dict(autorange="reversed"),
    bargap=0.25
)

fig1.show()


C:\Users\Aditya\AppData\Roaming\Python\Python313\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




### Growth vs Maintenance Stress in Aadhaar System

In [19]:
fig2 = px.scatter(
    state_data,
    x="total_enrolments",
    y="updates_per_100",
    hover_name="state",
    log_x=True,                 # spreads X properly
    opacity=0.6,                # overlap visibility
    size=[10]*len(state_data),  # fixed size = cleaner look
    color="updates_per_100",    # same vibe as before
    color_continuous_scale="Viridis",  # same family, not aggressive
    title="Growth vs Maintenance Stress in Aadhaar System",
    labels={
        "total_enrolments": "Total Aadhaar Enrolments (log scale)",
        "updates_per_100": "Updates per 100 Enrolments"
    }
)

# Add ONLY one reference line (keep it subtle)
fig2.add_hline(
    y=state_data['updates_per_100'].mean(),
    line_dash="dot",
    line_color="gray",
    annotation_text="National Average",
    annotation_position="top left"
)
fig2.update_traces(marker=dict(line=dict(width=0.5, color='white')))


fig2.show()


### Demographic Composition of Aadhaar Updates by State

In [20]:
# 1. Remove duplicate columns (critical)
df = demo.loc[:, ~demo.columns.duplicated()].copy()

# 2. Clean state column
df['state'] = df['state'].astype(str).str.strip()
df = df[df['state'].str.lower() != 'nan']

# 3. Keep only required columns
df = df[['state', 'demo_age_5_17', 'demo_age_17_']]

# 4. Aggregate by state
state_age = df.groupby('state', as_index=False).sum()

# 5. Take top 10 states by adult updates
state_age = state_age.sort_values(
    'demo_age_17_', ascending=False
).head(10)

# 6. Plot stacked bar (your original structure)
fig3 = px.bar(
    state_age,
    x='state',
    y=['demo_age_5_17', 'demo_age_17_'],
    title="Demographic Composition of Aadhaar Updates (Top States)",
    labels={'value': 'Number of Updates', 'state': 'State'},
    color_discrete_map={
        'demo_age_5_17': '#60a5fa',
        'demo_age_17_': '#f97316'
    }
)

# 7. Clean legend labels
fig3.for_each_trace(
    lambda t: t.update(
        name='Below 17' if t.name == 'demo_age_5_17' else '17 and Above'
    )
)

# 8. Readability tweaks
fig3.update_layout(
    xaxis_tickangle=-30,
    bargap=0.25
)

fig3.show()
